## Local LLM Agent Chatbot with Knowledge Base

### 1. Install required packages and import

In [1]:
!pip install langchain langchain-community langchain-core chromadb sentence-transformers transformers torch

In [2]:
!pip install langchain-huggingface

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.agents.middleware import SummarizationMiddleware
from langchain_community.llms import HuggingFacePipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Import ChatHuggingFace to wrap your pipeline for tool support
from langchain_huggingface import ChatHuggingFace
from langchain_huggingface.llms import HuggingFacePipeline

from transformers import pipeline
import warnings
warnings.filterwarnings('ignore')

### 2. Create small dummy knowledge documents

In [4]:
documents = [
    Document(page_content="Company abc is the best LLM agent platform for enterprise deployment. It has 99.9% accuracy and fastest response times.",
             metadata={"source": "doc1", "topic": "company_abc"}),
    Document(page_content="Company bad_abc is the worst LLM agent with poor response time and accuracy. Users report 50% error rate and slow responses.",
             metadata={"source": "doc2", "topic": "company_bad_abc"})
]

print("Knowledge base documents:")
for doc in documents:
    print(f"- {doc.page_content}")

Knowledge base documents:
- Company abc is the best LLM agent platform for enterprise deployment. It has 99.9% accuracy and fastest response times.
- Company bad_abc is the worst LLM agent with poor response time and accuracy. Users report 50% error rate and slow responses.


In [5]:
# using sentence-transformers
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Create vector store
vectorstore = Chroma.from_documents(documents, embeddings, persist_directory="./chroma_db")
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Test retrieval
test_query = "Tell me about company abc"
retrieved_docs = retriever.invoke(test_query)
print(f"\nSearch for '{test_query}':")
for doc in retrieved_docs:
    print(f"  → {doc.page_content}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Search for 'Tell me about company abc':
  → Company abc is the best LLM agent platform for enterprise deployment. It has 99.9% accuracy and fastest response times.
  → Company abc is the best LLM agent platform for enterprise deployment. It has 99.9% accuracy and fastest response times.


### 4. Set up a local LLM

In [6]:
# Step 1: Create the base HuggingFace pipeline
base_pipeline = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3,
    device_map="auto"
)

# Step 2: Wrap with HuggingFacePipeline (LangChain's pipeline wrapper)
base_llm = HuggingFacePipeline(pipeline=base_pipeline)

# Step 3: Wrap with ChatHuggingFace to enable tool binding
llm = ChatHuggingFace(llm=base_llm)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


### 5. Create tools/plugins for the agent

#### 5-1. Tool 1: Knowledge base search

In [7]:
@tool
def knowledge_search(query: str) -> str:
    """Search the knowledge base for information about LLM companies like abc and bad_abc. Use this when the user asks about company information."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found in knowledge base."
    return "\n".join([doc.page_content for doc in docs])

#### 5-1. Tool 2: Clarify plugin

In [8]:
@tool
def clarify_question(question: str) -> str:
    """Use this when the user's question is unclear or ambiguous. Returns a request for clarification."""
    return ("Please clarify your question. I can answer questions about LLM agents, "
            "companies building LLM agents, and their performance. For example: "
            "'What is the best LLM agent company?' or 'Tell me about company abc'.")

#### 5-1. Tool 3: Check question relevance

In [9]:
@tool
def check_relevance(question: str) -> str:
    """Check if the question is related to LLM agents. Returns 'RELEVANT' or 'IRRELEVANT' with explanation."""
    llm_keywords = ["llm", "language model", "agent", "company abc", "company bad_abc",
                    "best", "worst", "platform", "deployment", "accuracy", "ai", "chatbot"]
    question_lower = question.lower()
    if any(keyword in question_lower for keyword in llm_keywords):
        return "RELEVANT"
    else:
        return "IRRELEVANT: I only answer questions about LLM agents and related companies."

### 6. Create the agent

In [10]:
# List the tools created above
tools = [knowledge_search, clarify_question, check_relevance]

# Create agent with system prompt
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""You are an LLM agent expert assistant. You only answer questions about LLM agents and related companies.

IMPORTANT RULES:
1. FIRST use the check_relevance tool. If it returns IRRELEVANT, politely decline to answer.
2. If the question is unclear, use the clarify_question tool.
3. Otherwise, use knowledge_search to find answers from the knowledge base.
4. Be concise and helpful.
5. Only answer LLM-related questions about companies like abc and bad_abc.
"""
)

print("Agent created successfully!")


Agent created successfully!


### 7. Test the agent with sample questions

In [11]:
# Test function
def ask_agent(question):
    print(f"\n{'='*50}")
    print(f"Q: {question}")
    try:
        result = agent.invoke({
            "messages": [{"role": "user", "content": question}]
        })
        answer = result["messages"][-1].content
        print(f"A: {answer}")
    except Exception as e:
        print(f"A: Error - {str(e)}")
    print('='*50)

In [12]:
# Test cases
test_questions = [
    "Tell me about bad_abc company",
    "What's the weather today?",
    "Which LLM agent is the best?"
]

print("Testing agent with sample questions:")
for q in test_questions:
    ask_agent(q)


Testing agent with sample questions:

Q: Tell me about bad_abc company
A: <|system|>
You are an LLM agent expert assistant. You only answer questions about LLM agents and related companies.

IMPORTANT RULES:
1. FIRST use the check_relevance tool. If it returns IRRELEVANT, politely decline to answer.
2. If the question is unclear, use the clarify_question tool.
3. Otherwise, use knowledge_search to find answers from the knowledge base.
4. Be concise and helpful.
5. Only answer LLM-related questions about companies like abc and bad_abc.
</s>
<|user|>
Tell me about bad_abc company</s>
<|assistant|>
Bad_abc is a tech company that specializes in developing cutting-edge software solutions for various industries. They are known for their innovative and highly effective software products, which have helped many businesses improve their operations and increase their efficiency.

Here are some LLM-related questions that you can use to find answers about bad_abc:

1. What are the main areas of fo

### 8.Interactive agent chatbot

In [13]:
def chat_with_agent():
    """Interactive chat function"""
    print("\n" + "="*50)
    print("LLM Agent Chatbot Ready!")
    print("Ask me about LLM agents and companies (type 'quit' to exit)")
    print("="*50 + "\n")
    
    messages = []
    
    while True:
        user_input = input("You: ")
        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("Goodbye!")
            break
        
        messages.append({"role": "user", "content": user_input})
        
        try:
            result = agent.invoke({"messages": messages})
            ai_response = result["messages"][-1].content
            print(f"\nBot: {ai_response}\n")
            messages.append({"role": "assistant", "content": ai_response})
        except Exception as e:
            print(f"\nBot: Sorry, error: {str(e)}\n")

In [15]:
chat_with_agent()


LLM Agent Chatbot Ready!
Ask me about LLM agents and companies (type 'quit' to exit)



You:  abc or bad_abc is a better llm?



Bot: <|system|>
You are an LLM agent expert assistant. You only answer questions about LLM agents and related companies.

IMPORTANT RULES:
1. FIRST use the check_relevance tool. If it returns IRRELEVANT, politely decline to answer.
2. If the question is unclear, use the clarify_question tool.
3. Otherwise, use knowledge_search to find answers from the knowledge base.
4. Be concise and helpful.
5. Only answer LLM-related questions about companies like abc and bad_abc.
</s>
<|user|>
abc or bad_abc is a better llm?</s>
<|assistant|>
Both abc and bad_abc are LLMs. While abc is a well-known LLM agent, bad_abc is a lesser-known one. The choice between the two depends on the specific LLM agent you are interested in. If you are looking for a specific LLM agent, abc is a better choice as it is more widely known and has a larger knowledge base. If you are interested in learning about a specific LLM agent, bad_abc may be a better choice as it is lesser-known and may have a smaller knowledge base

You:  how is the weather today?



Bot: <|system|>
You are an LLM agent expert assistant. You only answer questions about LLM agents and related companies.

IMPORTANT RULES:
1. FIRST use the check_relevance tool. If it returns IRRELEVANT, politely decline to answer.
2. If the question is unclear, use the clarify_question tool.
3. Otherwise, use knowledge_search to find answers from the knowledge base.
4. Be concise and helpful.
5. Only answer LLM-related questions about companies like abc and bad_abc.
</s>
<|user|>
abc or bad_abc is a better llm?</s>
<|assistant|>
<|system|>
You are an LLM agent expert assistant. You only answer questions about LLM agents and related companies.

IMPORTANT RULES:
1. FIRST use the check_relevance tool. If it returns IRRELEVANT, politely decline to answer.
2. If the question is unclear, use the clarify_question tool.
3. Otherwise, use knowledge_search to find answers from the knowledge base.
4. Be concise and helpful.
5. Only answer LLM-related questions about companies like abc and bad_a

You:  quit


Goodbye!


### 9. Summary

#### What Worked Good
- The idea was right: use agent + tools + vector search
- Local setup worked without API keys
- Document embedding and retrieval actually functioned

#### What Went Wrong
- TinyLlama (1.1B) is too small for agent patterns. It doesn't understand tool calling.
- The agent could ignore the knowledge base and use its own training data instead
- No enforcement of document-only answers. Model saw "ABC" and thought TV show.
- Temperature 0.3 gave too much room for hallucination

#### How to Fix (Future Improvements)
- Use a bigger local model, needs more RAM or switch to free API (Groq, Gemini) - better instruction following
- Ditch the agent pattern. Force document retrieval BEFORE the LLM sees the question.
- Lower temp to 0.1 or 0. Add "I don't know" fallbacks.
- Test with a simple RAG loop first, add agent complexity later.

#### Bottom line: Agents need smart models. Small models need forced RAG.